# 37 — Havadan **RGB** görüntü için Stage C (VTUAV-VIS)

31'in RGB ikizi. 31 termal kalır; bu notebook ona dokunmaz.

**Tek kaynak: VTUAV-VIS.** Burada kullanılan maskeler yazarların kendi kare
başına instance maskeleri — bir teacher modelin ürettikleri değil. `vtuav`
tracking ve `birdsai` termal oldukları için bu koşuda hiç yer almıyor,
Anti-UAV410 ise zaten ters perspektif.

**Held-out gerçek.** `test_00x` arşivleri yazarların "kimse bunlarla eğitmesin"
dediği diziler. `split_flights(hold_out=...)` ile test seti o seçim oluyor,
dizi adlarının hash'i değil.

**Taban 35.** Adı "pretrain" ama yapısı Stage B: `METHOD="finetune"`, önce
head sonra encoder, RGB havuzlarında gerçek maske etiketleriyle. Yani buradan
Stage C'ye geçerken atlanan bir aşama yok.

**Stock burada alan içinde.** EdgeTAM RGB videoda (SA-V) eğitildi. Termalde
stock'u yenmek alan dışı bir modeli yenmekti; RGB'de değil. İki kollu precheck
ikisini de basar — RGB'de stock'a yenilmek termaldekinden daha az şaşırtıcıdır,
ve bunu sayı gelmeden önce bilmek gerekir.

In [ ]:
# --- Runtime -----------------------------------------------------------
import os, sys
from pathlib import Path

REPO   = Path("/content/sam-dedection")
BRANCH = "claude/thermal-stage-b-training-43ktcl"
if not REPO.exists():
    !git clone -q -b {BRANCH} https://github.com/yigitkayabagci/sam-dedection.git {REPO}
!git -C {REPO} fetch -q origin {BRANCH}
!git -C {REPO} checkout -q {BRANCH} && git -C {REPO} merge -q --ff-only origin/{BRANCH}
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!bash scripts/setup_edgetam.sh 2>&1 | tail -5
!pip install -q -r requirements.txt
!pip install -q gdown tqdm matplotlib
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
!python -m unittest -q tests.test_aerial_video tests.test_clip_loop \
    tests.test_training_losses tests.test_samurai tests.test_fetch_datasets \
    2>&1 | tail -3

## Ayarlar

VTUAV-VIS parçaları büyük: `train_001` 9.1 GB, `train_002` 16.1, `train_003`
17.9; test arşivleri 15-17.7 GB. Maskeler ~her 30. karede ve `--frames masked`
yalnız onları çıkarıyor, yani **bir eğitim + bir test arşivi** ilk koşu için
yeterli. Sekizini birden indirmek 126 GB ve Drive quota'sı demek.

`fetch_datasets` her parçayı **kendi klasörüne** açar ve dizi adları parçayı
taşır. Sebep isimlendirme: VTUAV-VIS dizileri hedef türü + parça başına
sıfırlanan bir sayaçla adlandırılıyor, yani `test_001.zip` içinde `train_003`
adlı bir dizi var ve `train_003.zip` bambaşka bir arşiv. Düz açılsalardı
kareleri aynı klasörde karışırdı.

Drive'da "Bir kopyasını oluştur" ile aldığınız arşivler `MyDrive/datasets/`
altında `test_001.zip adlı dosyanın kopyası` gibi adlanır; `staged()` artık
bu adları da tanıyor, yeniden adlandırmanız gerekmiyor.

In [ ]:
import errno, gc, json, math, shutil, subprocess, zipfile
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from tqdm.auto import tqdm

SIZE, CLIP_LEN, CLIP_STRIDE = 512, 8, 1
SEED = 0
MODALITY = "rgb"

VTUAV_VIS_TRAIN_PARTS = ["train_001", "train_002", "train_003"]
VTUAV_VIS_HOLD_OUT_PARTS = ["test_001", "test_002", "test_003",
                            "test_004", "test_005"]

SOURCE_WEIGHTS = {"vtuav_vis": 1.0}
TRAIN_CLIP_POOL = 6000
VAL_CLIP_POOL   = 800
TEST_CLIP_POOL  = 800
LOW_CONTRAST_QUANTILE = 0.40
LOW_CONTRAST_REPEAT   = 2
CONTRAST_AUDIT_STRIDE = 20

AUGMENT_PROB     = 0.75
GLOBAL_CONTRAST  = (0.35, 0.90)
TARGET_CONTRAST  = (0.15, 0.65)
BRIGHTNESS_SHIFT = (-0.05, 0.05)
SENSOR_NOISE     = (0.0, 0.018)
BLUR_PROB        = 0.20

STEPS_PER_EPOCH = 400
VAL_BATCHES     = 24
BATCH_CEILING   = 128
LOADER_WORKERS  = min(2 * (os.cpu_count() or 4), 24)
PREFETCH_DEPTH  = 2
EVAL_PER_SOURCE = 6
PRECHECK_PER_SOURCE = 4
PRECHECK_FRAMES = 240
PRECHECK_MIN_STATE_ACCURACY = 0.65
PRECHECK_MAX_LOST_RATE = 0.10
PRECHECK_MAX_LONGEST_DROPOUT = 24
PRECHECK_AGAINST_STOCK = True
STOP_AFTER_PRECHECK = False
INSPECT_DATA = True
INSPECT_PER_SOURCE = 6
INSPECT_SPAN = 12
INSPECT_WINDOWS = 2
PREVIEW_RUN = True
PREVIEW_FRAMES = 100
PREVIEW_FPS = 6
RENDER_BEFORE_AFTER = True
DEMO_FRAMES, DEMO_FPS, DEMO_PRE_ROLL = 400, 20, 80

DATA = Path("/content/data")
VTUAV_VIS_DATA = DATA / "VTUAV_VIS_rgb_stage_c"
WORK = Path("/content/work/aerial_rgb_tracking")
for directory in (DATA, VTUAV_VIS_DATA, WORK):
    directory.mkdir(parents=True, exist_ok=True)

from google.colab import drive
drive.mount("/content/drive")
DATASETS_DRIVE = Path("/content/drive/MyDrive/datasets")
STAGE_B_DRIVE = Path("/content/drive/MyDrive/edgetam-stage-b")
BASE_STAGE_B_OVERRIDE = ""
BASE_STAGE_B_CANDIDATES = [
    STAGE_B_DRIVE / "pretrain_rgb_aerial"
                  / "edgetam_pool_pretrain_rgb_aerial_512.pt",
]
BASE_STAGE_B = (Path(BASE_STAGE_B_OVERRIDE) if BASE_STAGE_B_OVERRIDE
                else next((path for path in BASE_STAGE_B_CANDIDATES
                           if path.is_file()),
                          BASE_STAGE_B_CANDIDATES[0]))
assert BASE_STAGE_B.is_file(), (
    f"RGB Stage-B checkpoint yok: {BASE_STAGE_B}. Önce notebook 35'i "
    f"bitirin, ya da BASE_STAGE_B_OVERRIDE'a tam yolu yazın.")
BASE_TAG = (Path(BASE_STAGE_B).stem
            .replace("edgetam_pool_", "").replace(f"_{SIZE}", ""))[:48]
MIRROR = Path("/content/drive/MyDrive/edgetam-stage-c") / (
    f"aerial_rgb_tracking_from_{BASE_TAG}")
MIRROR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = REPO / (
    f"checkpoints/edgetam_aerial_rgb_tracking_from_{BASE_TAG}_{SIZE}.pt")
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print("Stage-B base:", BASE_STAGE_B)
print("Stage-C output:", MIRROR)

## Veriyi getir

Her parça tek tek çekilir ve `.done` işareti bırakır: bir parça Drive
quota'sına takılırsa yeniden koşmak yalnız eksik olana mal olur. Drive'daki
kendi kopyalarınız `MyDrive/datasets/` altındaysa `staged()` onları ağa
gitmeden bulur -- kopyalama sırasında aldıkları "... kopyası" adlarıyla
birlikte.

In [ ]:
VIS_STAGED = VTUAV_VIS_DATA / "_staged"
VIS_STAGED.mkdir(parents=True, exist_ok=True)

def fetch_vis(part):
    marker = VIS_STAGED / f"{part}.done"
    if marker.is_file():
        print("already staged", part)
        return
    subprocess.run([
        sys.executable, "tools/fetch_datasets.py", "vtuav_vis",
        "--dest", str(VTUAV_VIS_DATA), "--parts", part,
        "--frames", "masked"], check=True)
    marker.write_text("ok\n")

for _part in VTUAV_VIS_TRAIN_PARTS + VTUAV_VIS_HOLD_OUT_PARTS:
    fetch_vis(_part)

for _part in VTUAV_VIS_TRAIN_PARTS + VTUAV_VIS_HOLD_OUT_PARTS:
    _where = VTUAV_VIS_DATA / _part
    assert _where.is_dir(), (
        f"{_where} yok. `fetch_datasets` her parçayı kendi klasörüne açar; "
        f"bu klasör yoksa parça inmemiştir.")
    print(f"{_part:<12}{len(list(_where.rglob('mask/' + MODALITY))):>4} "
          f"maskeli dizi")

## Split: yazarların held-out'u, hash değil

Eğitim parçaları kendi içlerinde train/val'e bölünür; test seti **tamamen**
`test_00x` arşivlerinden gelir. `split_flights(hold_out=...)` verilen test
setini örneklenenin yerine koyar — ikisini karıştırmak, çıkan sayının hangi
yarıdan geldiğini geri alınamaz hale getirirdi.

Kesişme kontrolü burada yapılır ve sessiz geçilmez: iki taraf aynı dizi adını
taşıyorsa held-out held-out değildir.

In [ ]:
from src.training import (
    Sequence, SequenceLabels, empty_stores, flight_name, source_name,
    split_flights, video_clips, vtuav_vis_sequences, weighted_clip_sample,
)

def vis_from(parts):
    rows, stores = [], {}
    for part in parts:
        part_rows, part_stores = vtuav_vis_sequences(
            VTUAV_VIS_DATA / part, modality=MODALITY)
        rows += part_rows
        stores.update(part_stores)
    return rows, stores

train_sequences, TRAIN_STORES = vis_from(VTUAV_VIS_TRAIN_PARTS)
held_sequences, HELD_STORES = vis_from(VTUAV_VIS_HOLD_OUT_PARTS)
sequences = train_sequences + held_sequences
VIS_STORES = {**TRAIN_STORES, **HELD_STORES}

assert train_sequences, "eğitim parçalarından hiç dizi çıkmadı"
assert held_sequences, "held-out parçalarından hiç dizi çıkmadı"
_overlap = ({row.name for row in train_sequences}
            & {row.name for row in held_sequences})
assert not _overlap, (
    f"held-out ve eğitim aynı dizi adını paylaşıyor: {sorted(_overlap)}. "
    f"Parçalar kendi klasörlerine açılmadıysa adlar parçayı taşımaz ve "
    f"held-out held-out değildir.")

SPLITS = split_flights(train_sequences, seed=SEED, val_fraction=0.20,
                       test_fraction=0.0, hold_out=held_sequences)
print(f"{'part':<8}{'diziler':>9}{'kare':>10}{'maskeli':>10}")
for _part, _rows in SPLITS.items():
    print(f"{_part:<8}{len(_rows):>9}{sum(len(r) for r in _rows):>10}"
          f"{sum(len(VIS_STORES.get(r.name, {})) for r in _rows):>10}")
assert all(SPLITS.values()), "split'lerden biri boş."

# 31 turns the object-score term off when nothing in the mix is ever absent,
# and this run is exactly that case: `vtuav_vis_sequences` sets
# `exist=np.ones`, so every frame has its target. Counted rather than assumed,
# because the switch downstream reads the count and a hard-coded zero here
# would be a claim instead of a measurement.
ABSENT_FRAMES = {"vtuav_vis": sum(len(row) - int(row.labels.exist.sum())
                                  for row in sequences)}
TOTAL_ABSENT = sum(ABSENT_FRAMES.values())
print("absent (exist=False) frames per source:", ABSENT_FRAMES)

## Gerçek yerel kontrast ve clip havuzu

Tek kaynak olduğu için kaynaklar arası ağırlıklandırma burada bir şey yapmıyor;
kalan iş düşük kontrastı ayakta tutmak. Diziler kontrast sırasına sokulur ve
alt %40 aday havuzuna iki kez girer, böylece parlak gündüz uçuşları havuzu
tek başına doldurmaz.

In [ ]:
def frame_local_contrast(sequence, index):
    image = cv2.imread(str(sequence.frames[index]), cv2.IMREAD_GRAYSCALE)
    box = sequence.labels.boxes[index]
    if image is None or not np.isfinite(box).all():
        return np.nan
    x0, y0, x1, y1 = box
    height, width = image.shape
    side = max(x1 - x0, y1 - y0, 2)
    pad = max(6, int(round(side)))
    ax0, ay0 = max(0, int(x0) - pad), max(0, int(y0) - pad)
    ax1, ay1 = min(width, int(np.ceil(x1)) + pad), min(height, int(np.ceil(y1)) + pad)
    tx0, ty0 = max(0, int(x0)), max(0, int(y0))
    tx1, ty1 = min(width, int(np.ceil(x1))), min(height, int(np.ceil(y1)))
    target = image[ty0:ty1, tx0:tx1].astype(np.float32)
    patch = image[ay0:ay1, ax0:ax1].astype(np.float32)
    if target.size < 4 or patch.size <= target.size:
        return np.nan
    ring = np.ones(patch.shape, dtype=bool)
    ring[ty0 - ay0:ty1 - ay0, tx0 - ax0:tx1 - ax0] = False
    background = patch[ring]
    return (float(abs(target.mean() - background.mean()) /
                  max(background.std(), 3.0))
            if background.size >= 8 else np.nan)

def sequence_contrast(sequence):
    visible = sequence.labels.visible_indices()[::CONTRAST_AUDIT_STRIDE]
    values = [frame_local_contrast(sequence, int(index)) for index in visible]
    values = np.asarray([value for value in values if np.isfinite(value)])
    return float(np.median(values)) if values.size else np.nan

CONTRAST = {row.name: sequence_contrast(row) for row in sequences}
LOW_NAMES = set()
_rows = [row for row in SPLITS["train"] if np.isfinite(CONTRAST[row.name])]
_rows.sort(key=lambda row: CONTRAST[row.name])
LOW_NAMES.update(row.name for row in
                 _rows[:max(1, math.ceil(len(_rows) * LOW_CONTRAST_QUANTILE))])
print("low-contrast cut",
      CONTRAST[_rows[max(0, math.ceil(len(_rows) * LOW_CONTRAST_QUANTILE) - 1)].name]
      if _rows else "n/a")

def make_clips(rows, jitter):
    return video_clips(rows, length=CLIP_LEN, stride=1, size=SIZE,
                       min_visible=2, jitter=jitter, seed=SEED)

raw_train = make_clips(SPLITS["train"], jitter=32)
raw_train += [clip for clip in raw_train
              if clip.sequence.name in LOW_NAMES] * (LOW_CONTRAST_REPEAT - 1)
raw_val = make_clips(SPLITS["val"], jitter=0)
raw_test = make_clips(SPLITS["test"], jitter=0)
TRAIN_CLIPS = weighted_clip_sample(
    raw_train, SOURCE_WEIGHTS, TRAIN_CLIP_POOL, SEED)
VAL_CLIPS = weighted_clip_sample(raw_val, SOURCE_WEIGHTS, VAL_CLIP_POOL, SEED + 1)
TEST_CLIPS = weighted_clip_sample(raw_test, SOURCE_WEIGHTS, TEST_CLIP_POOL, SEED + 2)

STORES = empty_stores(sequences)
STORES.update(VIS_STORES)

if INSPECT_DATA:
    from tools.inspect_stage_c import render
    print("\ncontact sheets -- what Stage C is about to train on:")
    render(sequences, STORES, MIRROR / "inspect",
           per_source=INSPECT_PER_SOURCE, span=INSPECT_SPAN,
           windows=INSPECT_WINDOWS)

if PREVIEW_RUN:
    from tools.preview_sequence import pick_sequence, preview
    _watch = pick_sequence(sequences, STORES)
    if _watch is not None:
        print(f"\nizlenecek dizi: {_watch.name} "
              f"({len(STORES.get(_watch.name, {}))} maskeli kare)")
        PREVIEW_REPORT = preview(_watch, STORES[_watch.name],
                                 MIRROR / "preview", count=PREVIEW_FRAMES,
                                 fps=PREVIEW_FPS)
        from IPython.display import Video, display
        _mp4 = Path(PREVIEW_REPORT["video"]) if PREVIEW_REPORT.get("video") else None
        if _mp4 is not None and _mp4.is_file():
            display(Video(str(_mp4), embed=True, width=900))

MASK_FRAMES = sum(len(STORES[row.name]) for row in sequences)
VISIBLE_FRAMES = sum(int(row.labels.exist.sum()) for row in sequences)
print(f"\nmask supervision  {MASK_FRAMES}/{VISIBLE_FRAMES} visible frames "
      f"({MASK_FRAMES / max(VISIBLE_FRAMES, 1):.1%})")
print("!! VTUAV-VIS'te `exist` her karede True: bu koşuda kaybolma denetimi "
      "YOK,\n   yani object-score kafası hiçbir şey öğrenmiyor. 31'in vtuav "
      "kolu gerçek\n   `exist` taşıyan tek kaynaktır.")

for _label, _clips in (("train", TRAIN_CLIPS), ("val", VAL_CLIPS),
                       ("test", TEST_CLIPS)):
    print(_label, len(_clips))

## Stage C gerçekten gerekli mi? — eğitim öncesi video audit'i

Stage B yalnız tek kare görür; bu nedenle statik IoU iyi olsa bile bir kez
yanlış maskeyi belleğe yazınca takipçiyi geri getirmeyi öğrenmiş değildir.
Yine de pahalı video eğitimini körlemesine başlatmıyoruz: her kaynaktan en
düşük kontrastlı iki validation track'inin en zor 240 karelik bölümü önce
yalnız Stage B ile çalıştırılır.

Operasyonel kırmızı bayraklar: State Accuracy `<0.65`, kayıp-kare oranı
`>10%` veya en uzun dropout `>24` kare. Bunlar evrensel benchmark eşikleri
değil; tek-prompt uzun video kullanımında yeniden-prompt maliyetini görünür
yapan başlangıç sınırlarıdır. Herhangi bir kaynak bu sınırı aşıyorsa Stage C
önerilir. Kullanımınız her karede detector ile yeniden başlatıyorsa bu audit
iyi sonuç verdiğinde Stage C opsiyonel olabilir.

In [ ]:
from src.accuracy import score_sequence
from src.trackers import build_tracker
from tools.eval_antiuav import track_sequence

def precheck_segment(sequence):
    visible = sequence.labels.visible_indices()[::max(CONTRAST_AUDIT_STRIDE // 2, 1)]
    measured = [(int(index), frame_local_contrast(sequence, int(index)))
                for index in visible]
    measured = [(index, value) for index, value in measured if np.isfinite(value)]
    hard = min(measured, key=lambda pair: pair[1])[0] if measured else int(visible[0])
    begin = max(0, hard - PRECHECK_FRAMES // 2)
    end = min(len(sequence), begin + PRECHECK_FRAMES)
    begin = max(0, end - PRECHECK_FRAMES)
    return Sequence(
        name=f"{sequence.name}__stage_c_precheck",
        split=sequence.split,
        frames=sequence.frames[begin:end],
        labels=SequenceLabels(
            exist=sequence.labels.exist[begin:end].copy(),
            boxes=sequence.labels.boxes[begin:end].copy()))

def precheck_candidates():
    chosen = []
    for source in sorted({source_name(row) for row in SPLITS["val"]}):
        rows = [row for row in SPLITS["val"]
                if source_name(row) == source
                and row.labels.visible_indices().size]
        rows.sort(key=lambda row: (not np.isfinite(CONTRAST[row.name]),
                                   CONTRAST[row.name]))
        chosen += rows[:PRECHECK_PER_SOURCE]
    return chosen

STOCK_CKPT = REPO / "third_party/EdgeTAM/checkpoints/edgetam.pt"
PRECHECK_ARMS = [("stage_b", BASE_STAGE_B)]
if PRECHECK_AGAINST_STOCK and STOCK_CKPT.is_file():
    PRECHECK_ARMS.insert(0, ("stock", STOCK_CKPT))
elif PRECHECK_AGAINST_STOCK:
    print("!! stock checkpoint yok:", STOCK_CKPT,
          "-- yalnız stage_b ölçülecek.")

_segments = [precheck_segment(_row) for _row in precheck_candidates()]

def run_precheck(label, checkpoint):
    config = {
        "model_cfg": "configs/edgetam.yaml", "checkpoint": str(checkpoint),
        "image_size": SIZE, "device": "cuda", "precision": "bfloat16",
        "mask_threshold": 0.0, "offload_video_to_cpu": False,
        "offload_state_to_cpu": False,
    }
    tracker = build_tracker("edgetam", **config)
    rows = []
    try:
        for segment in tqdm(_segments, desc=f"{label} video precheck"):
            pred, gt = track_sequence(tracker, segment, "crop", SIZE)
            score = score_sequence(segment.name, pred, gt)
            lost = sum(score.dropouts.lengths)
            name = segment.name.replace("__stage_c_precheck", "")
            rows.append({
                "arm": label, "name": name, "source": source_name(name),
                "frames": score.frames,
                "state_accuracy": score.state_accuracy,
                "success_auc": score.success_auc,
                "lost_frames": lost,
                "lost_rate": lost / max(score.frames, 1),
                "longest_dropout": max(score.dropouts.lengths, default=0),
                "sequence_contrast": CONTRAST[name],
            })
    finally:
        tracker.reset(); del tracker
        gc.collect(); torch.cuda.empty_cache()
    return rows

PRECHECK_ARM_ROWS = {label: run_precheck(label, checkpoint)
                     for label, checkpoint in PRECHECK_ARMS}
PRECHECK_ROWS = PRECHECK_ARM_ROWS["stage_b"]

# Bir kol diğerini yenmiş mi: `state_accuracy` hedefin var/yok
# durumunu ne kadar doğru bildiğidir, yani kaybolup geri gelmenin
# ölçüsü; `longest_dropout` da geri gelene kadar geçen en uzun süre.
PRECHECK_DELTAS = []
if "stock" in PRECHECK_ARM_ROWS:
    _stock_by_name = {row["name"]: row for row in PRECHECK_ARM_ROWS["stock"]}
    for _row in PRECHECK_ROWS:
        _base = _stock_by_name.get(_row["name"])
        if _base is None:
            continue
        PRECHECK_DELTAS.append({
            "name": _row["name"], "source": _row["source"],
            "state_accuracy": _row["state_accuracy"] - _base["state_accuracy"],
            "success_auc": _row["success_auc"] - _base["success_auc"],
            "lost_rate": _row["lost_rate"] - _base["lost_rate"],
            "longest_dropout": _row["longest_dropout"] - _base["longest_dropout"],
        })
    print(f"\n{'source':<14}{'arm':<9}{'SA':>8}{'AUC':>8}{'lost%':>9}"
          f"{'longest':>10}")
    for _row in PRECHECK_ROWS:
        for _label in ("stock", "stage_b"):
            _one = (_stock_by_name[_row["name"]] if _label == "stock"
                    else _row)
            print(f"{_one['source'] if _label == 'stock' else '':<14}"
                  f"{_label:<9}{_one['state_accuracy']:>8.3f}"
                  f"{_one['success_auc']:>8.3f}{_one['lost_rate']:>9.1%}"
                  f"{_one['longest_dropout']:>10}")
    _mean = lambda key: (sum(row[key] for row in PRECHECK_DELTAS)
                         / max(len(PRECHECK_DELTAS), 1))
    _sa, _lost = _mean("state_accuracy"), _mean("lost_rate")
    print(f"\nstage_b - stock:  state_accuracy {_sa:+.4f}   "
          f"lost_rate {_lost:+.1%}")
    PRECHECK_STOCK_WINS = _sa < 0 or _lost > 0
    if PRECHECK_STOCK_WINS:
        print("!! Stage B, bu klipler üzerinde stock'tan DAHA KÖTÜ takip "
              "ediyor.\n"
              "   Bu, 'Stage C'ye ihtiyaç var mı' sorusundan farklı bir "
              "bulgudur: eğitim\n"
              "   tek-kare maskeyi iyileştirirken video sürekliliğini "
              "bozmuş demektir.\n"
              "   Beklenen mekanizma: memory_attention/memory_encoder "
              "donuk (finetune.py\n"
              "   FROZEN_MODULES) ve stock encoder özelliklerine göre "
              "eğitilmiş; encoder\n"
              "   o dağılımdan uzaklaştıkça bellek eğitilmediği "
              "özellikleri okuyor.")
else:
    PRECHECK_STOCK_WINS = None

PRECHECK_REASONS = []
for _row in PRECHECK_ROWS:
    _flags = []
    if _row["state_accuracy"] < PRECHECK_MIN_STATE_ACCURACY:
        _flags.append("low_state_accuracy")
    if _row["lost_rate"] > PRECHECK_MAX_LOST_RATE:
        _flags.append("high_lost_rate")
    if _row["longest_dropout"] > PRECHECK_MAX_LONGEST_DROPOUT:
        _flags.append("long_dropout")
    if _flags:
        PRECHECK_REASONS.append({"name": _row["name"], "flags": _flags})
PRECHECK_RECOMMEND_STAGE_C = bool(PRECHECK_REASONS)
if "stock" not in PRECHECK_ARM_ROWS:
    print(f"\n{'source':<14}{'SA':>8}{'AUC':>8}{'lost%':>9}{'longest':>10}")
    for _row in PRECHECK_ROWS:
        print(f"{_row['source']:<14}{_row['state_accuracy']:>8.3f}"
              f"{_row['success_auc']:>8.3f}{_row['lost_rate']:>9.1%}"
              f"{_row['longest_dropout']:>10}")
print("\nStage C recommendation:",
      "RUN — video continuity is below the gate"
      if PRECHECK_RECOMMEND_STAGE_C else
      "OPTIONAL — Stage B passed this small low-contrast audit")
if PRECHECK_REASONS:
    print("reasons:", PRECHECK_REASONS)
PRECHECK_REPORT = {
    "base_stage_b": str(BASE_STAGE_B),
    "arms": {label: str(path) for label, path in PRECHECK_ARMS},
    "arm_rows": PRECHECK_ARM_ROWS,
    "deltas": PRECHECK_DELTAS,
    "stock_wins": PRECHECK_STOCK_WINS,
    "thresholds": {
        "min_state_accuracy": PRECHECK_MIN_STATE_ACCURACY,
        "max_lost_rate": PRECHECK_MAX_LOST_RATE,
        "max_longest_dropout": PRECHECK_MAX_LONGEST_DROPOUT,
    },
    "recommend_stage_c": PRECHECK_RECOMMEND_STAGE_C,
    "reasons": PRECHECK_REASONS, "rows": PRECHECK_ROWS,
}
(MIRROR / "stage_c_precheck.json").write_text(
    json.dumps(PRECHECK_REPORT, indent=2) + "\n")
if STOP_AFTER_PRECHECK:
    raise SystemExit(
        "STOP_AFTER_PRECHECK=True: stage_c_precheck.json yazıldı; "
        "kararı inceleyip devam etmek için False yapın.")

## Stage-B checkpoint'ini video belleği açık halde devam eğit

Teacher forcing yoktur: her kare, modelin önceki kendi tahminini belleğe
yazdığı gerçek deployment yolundan geçer. Bellek attention/encoder sabit
kalır; mask decoder, IoU/object score ve görüntü encoder'ı düşük hızla
uyarlanır.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor
from src.trackers._hydra_overrides import image_size_overrides
from src.training.finetune import Rates, apply_freeze, summarise_freeze

model = build_sam2_video_predictor(
    "configs/edgetam.yaml", str(BASE_STAGE_B), device="cuda",
    hydra_overrides_extra=image_size_overrides(SIZE))
model.eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params")
print(summarise_freeze(apply_freeze(model, "encoder"), model))

trainable = {name for name, parameter in model.named_parameters()
             if parameter.requires_grad}
assert not any("memory_attention" in name or "memory_encoder" in name
               for name in trainable), "memory coordinates must stay frozen"

## Zamanda yumuşak düşük-kontrast dönüşümü

Parlaklık/kontrast kareden kareye rastgele titreştirilmez; klibin başı ile
sonu arasında düzgün geçiş yapılır. Hedef kutusu çevre halkasının termal
tonuna yaklaştırılır. Validation ve test bu dönüşümü görmez.

In [ ]:
from dataclasses import dataclass, replace
import torch.nn.functional as F
from src.training.antiuav import MEAN, STD
from src.training.clip_loop import clip_losses
from src.training.losses import Weights
from src.training.schedule import CLIPS, Loop

# `object_score` ölçülen kayboluş sayısına bakıyor, sabite değil.
# Terim `losses.frame_loss`'ta her kareye koşulsuz uygulanıyor
# (kaynağın gerçek `exist` taşıyıp taşımadığına bakan bir kapı yok),
# yani hiç `exist=False` kare yokken 2.0 ağırlık "her karede hedef
# var" diyen sabite karşı BCE demektir. `losses.py`'nin kendi modül
# docstring'i sonucunu yazıyor: öyle bir başlık koşulsuz ateşlemeyi
# öğrenir, `object_score_logits` negatife düşünce EdgeTAM bankaya
# `no_obj_ptr` yazdığı ve sonraki kareler onu geri okuduğu için kendi
# belleğini zehirler. Stage B aynı gerekçeyle terimi hiç kullanmıyor.
OBJECT_SCORE_WEIGHT = 2.0 if TOTAL_ABSENT else 0.0
print(f"object_score weight {OBJECT_SCORE_WEIGHT} "
      f"({TOTAL_ABSENT} absent frames measured)")
if not TOTAL_ABSENT:
    print("   Açık hiçbir kaynak `exist=False` kare taşımıyor, o yüzden "
          "terim kapatıldı.\n"
          "   Bunun anlamı: bu koşu 'hedef gerçekten orada mı' "
          "başlığına hiçbir şey öğretmiyor.\n"
          "   Gerçek kayboluş yalnız VTUAV tracking'in ir.txt'inden "
          "gelir (USE_VTUAV=True).")
TRACK_WEIGHTS = Weights(focal=20.0, dice=1.0, iou=2.0,
                        object_score=OBJECT_SCORE_WEIGHT,
                        box_projection=1.0)

def uniform(shape, limits, device, generator):
    low, high = (float(value) for value in limits)
    return low + (high - low) * torch.rand(
        shape, device=device, generator=generator)

def temporal_pair(batch, limits, generator, device):
    shape = (batch, 1, 1, 1, 1)
    return (uniform(shape, limits, device, generator),
            uniform(shape, limits, device, generator))

def low_contrast_augment(batch, generator):
    images = batch.images
    device = images.device
    count, frames, _, height, width = images.shape
    mean = torch.as_tensor(MEAN, device=device).view(1, 1, 3, 1, 1)
    std = torch.as_tensor(STD, device=device).view(1, 1, 3, 1, 1)
    gray = (images * std + mean).clamp(0, 1).mean(2, keepdim=True)
    chosen = (torch.rand((count, 1, 1, 1, 1), device=device,
                         generator=generator) < AUGMENT_PROB).float()
    phase = torch.linspace(0, 1, frames, device=device).view(1, frames, 1, 1, 1)

    c0, c1 = temporal_pair(count, GLOBAL_CONTRAST, generator, device)
    contrast = c0 + (c1 - c0) * phase
    center = gray.mean((-2, -1), keepdim=True)
    augmented = center + contrast * (gray - center)
    b0, b1 = temporal_pair(count, BRIGHTNESS_SHIFT, generator, device)
    augmented = augmented + b0 + (b1 - b0) * phase

    boxes = batch.boxes
    xx = torch.arange(width, device=device).view(1, 1, 1, width)
    yy = torch.arange(height, device=device).view(1, 1, height, 1)
    x0, y0, x1, y1 = [boxes[..., index].unsqueeze(-1).unsqueeze(-1)
                      for index in range(4)]
    valid = batch.exist.bool().unsqueeze(-1).unsqueeze(-1)
    inside = valid & (xx >= x0) & (xx < x1) & (yy >= y0) & (yy < y1)
    pad = torch.maximum((x1 - x0).abs().clamp(min=4),
                        (y1 - y0).abs().clamp(min=4))
    outer = valid & (xx >= x0 - pad) & (xx < x1 + pad) & \
            (yy >= y0 - pad) & (yy < y1 + pad)
    ring = (outer & ~inside).unsqueeze(2)
    ring_mean = ((gray * ring).sum((-2, -1), keepdim=True) /
                 ring.sum((-2, -1), keepdim=True).clamp(min=1))
    t0, t1 = temporal_pair(count, TARGET_CONTRAST, generator, device)
    factor = t0 + (t1 - t0) * phase
    suppressed = ring_mean + factor * (augmented - ring_mean)
    soft = F.avg_pool2d(inside.float().reshape(count * frames, 1, height, width),
                        11, stride=1, padding=5)
    soft = soft.reshape(count, frames, 1, height, width).clamp(0, 1) * chosen
    augmented = augmented * (1 - soft) + suppressed * soft

    sigma = uniform((count, 1, 1, 1, 1), SENSOR_NOISE, device, generator)
    augmented += chosen * sigma * torch.randn(
        gray.shape, device=device, generator=generator)
    blurred = F.avg_pool2d(augmented.reshape(count * frames, 1, height, width),
                           3, stride=1, padding=1).reshape(
                               count, frames, 1, height, width)
    blur = ((torch.rand((count, 1, 1, 1, 1), device=device,
                        generator=generator) < BLUR_PROB).float() * chosen)
    augmented = augmented * (1 - blur) + blurred * blur
    gray = (gray * (1 - chosen) + augmented * chosen).clamp(0, 1)
    return replace(batch, images=(gray.expand(-1, -1, 3, -1, -1) - mean) / std)

@dataclass(frozen=True)
class ContrastSplit:
    clips: list
    stores: dict
    augment: bool

def contrast_stream(split, batch, seed, limit, device="cuda", workers=8, depth=2):
    stream = CLIPS.stream(split, batch, seed, limit, device, workers, depth)
    generator = torch.Generator(device=device)
    generator.manual_seed(SEED if seed is None else int(seed))
    for item in stream:
        yield low_contrast_augment(item, generator) if split.augment else item

def tracking_loss(active_model, batch):
    return clip_losses(active_model, batch, weights=TRACK_WEIGHTS)

CONTRAST_LOOP = Loop(stream=contrast_stream, loss=tracking_loss,
                     val_loss=tracking_loss)
TRAIN_SPLIT = ContrastSplit(TRAIN_CLIPS, STORES, True)
VAL_SPLIT = ContrastSplit(VAL_CLIPS, STORES, False)

In [ ]:
from src.training.loader import auto_batch_size

apply_freeze(model, "encoder")
BATCH = auto_batch_size(
    model, TRAIN_CLIPS[:BATCH_CEILING], STORES, device="cuda",
    maximum=BATCH_CEILING, reserve=0.15)
ACCUM = max(1, math.ceil(32 / BATCH))
print("batch", BATCH, "accum", ACCUM, "effective", BATCH * ACCUM)

preview = next(contrast_stream(VAL_SPLIT, min(3, BATCH), SEED, 1))
changed = low_contrast_augment(
    preview, torch.Generator(device="cuda").manual_seed(1234))
mean = torch.as_tensor(MEAN, device="cuda").view(1, 1, 3, 1, 1)
std = torch.as_tensor(STD, device="cuda").view(1, 1, 3, 1, 1)
before = (preview.images * std + mean).clamp(0, 1).cpu()
after = (changed.images * std + mean).clamp(0, 1).cpu()
fig, axes = plt.subplots(2, before.shape[0], figsize=(4 * before.shape[0], 7))
axes = np.asarray(axes).reshape(2, -1)
for index in range(before.shape[0]):
    axes[0, index].imshow(before[index, 0].permute(1, 2, 0))
    axes[1, index].imshow(after[index, 0].permute(1, 2, 0))
    axes[0, index].axis("off"); axes[1, index].axis("off")
axes[0, 0].set_ylabel("real"); axes[1, 0].set_ylabel("augmented")
plt.tight_layout(); plt.show()
del preview, changed, before, after
gc.collect(); torch.cuda.empty_cache()

## Eğitim

İlk aşama decoder/object-score başlığını, ikinci aşama termal görüntü
özelliklerini düşük hızla açar. Checkpoint yalnız gerçek, augmentasyonsuz
validation loss iyileştiğinde kaydedilir.

In [ ]:
from src.training.finetune import save_checkpoint
from src.training.schedule import Schedule, run_stages

schedule = Schedule(
    stages=(("head", 2, Rates(head=5e-5)),
            ("encoder", 4, Rates(head=2e-5, neck=2e-5, trunk=5e-6))),
    batch=BATCH, accum=ACCUM, steps_per_epoch=STEPS_PER_EPOCH,
    val_batches=VAL_BATCHES, workers=LOADER_WORKERS,
    depth=PREFETCH_DEPTH, seed=SEED, patience=2,
    meta={
        "method": "aerial_temporal_contrast_finetune",
        "base": str(BASE_STAGE_B), "image_size": SIZE,
        "sources": SOURCE_WEIGHTS, "modality": MODALITY,
        "vtuav_vis_train_parts": VTUAV_VIS_TRAIN_PARTS,
        "vtuav_vis_hold_out_parts": VTUAV_VIS_HOLD_OUT_PARTS,
        "absent_frames": ABSENT_FRAMES,
        "object_score_weight": OBJECT_SCORE_WEIGHT,
        "base_stage_b_tag": BASE_TAG,
        "mask_frames": MASK_FRAMES,
        "mask_frames": MASK_FRAMES, "visible_frames": VISIBLE_FRAMES,
        "drawn_mask_frames": sum(len(store) for store in VIS_STORES.values()),
        "hold_out": VTUAV_VIS_HOLD_OUT_PARTS,
        "augment": {"prob": AUGMENT_PROB,
                    "global_contrast": GLOBAL_CONTRAST,
                    "target_contrast": TARGET_CONTRAST,
                    "brightness": BRIGHTNESS_SHIFT,
                    "noise": SENSOR_NOISE, "blur_prob": BLUR_PROB},
        "loss_weights": TRACK_WEIGHTS.__dict__,
    })
result = run_stages(
    model, TRAIN_SPLIT, VAL_SPLIT, schedule, freeze=apply_freeze,
    save=lambda active_model, meta: save_checkpoint(active_model, CHECKPOINT, meta),
    progress=lambda stream, total, desc: tqdm(stream, total=total, desc=desc),
    loop=CONTRAST_LOOP)
assert CHECKPOINT.is_file(), "training produced no checkpoint"
print("best validation clip loss", result["best_val_loss"], "->", CHECKPOINT)
del model
gc.collect(); torch.cuda.empty_cache()

## Tracking A/B — kaynak bazında

Stage B, yeni checkpoint ve yeni checkpoint+SAMURAI aynı held-out
uçuşlarda çalışır. Seçim validation toplam State Accuracy ile yapılır;
ayrılmış test uçuşlarına yalnız seçilen final kolu bir kez gider.

In [ ]:
from src.accuracy import score_sequence
from src.trackers import build_tracker
from tools.eval_antiuav import track_sequence

SAMURAI = {"enabled": True, "kf_weight": 0.15,
           "stable_frames": 15, "stable_iou": 0.3,
           "memory_iou": 0.5, "memory_obj_score": 0.0,
           "memory_kf_score": 0.0}

def tracker_config(checkpoint, samurai=None):
    config = {
        "model_cfg": "configs/edgetam.yaml", "checkpoint": str(checkpoint),
        "image_size": SIZE, "device": "cuda", "precision": "bfloat16",
        "mask_threshold": 0.0, "offload_video_to_cpu": False,
        "offload_state_to_cpu": False,
    }
    if samurai is not None:
        config["samurai"] = samurai
    return config

CONFIGS = {
    "stage_b": tracker_config(BASE_STAGE_B),
    "aerial_temporal": tracker_config(CHECKPOINT),
    "aerial_temporal+samurai": tracker_config(CHECKPOINT, SAMURAI),
}

def eval_subset(rows):
    selected = []
    for source in sorted({source_name(row) for row in rows}):
        seen_flights = set()
        for row in rows:
            if source_name(row) != source or flight_name(row) in seen_flights:
                continue
            selected.append(row)
            seen_flights.add(flight_name(row))
            if len(seen_flights) >= EVAL_PER_SOURCE:
                break
    return selected

def run_arm(label, rows):
    tracker = build_tracker("edgetam", **CONFIGS[label])
    output = []
    try:
        for sequence in tqdm(rows, desc=label):
            pred, gt = track_sequence(tracker, sequence, "crop", SIZE)
            score = score_sequence(sequence.name, pred, gt)
            output.append({
                "name": sequence.name, "source": source_name(sequence),
                "frames": score.frames, "state_accuracy": score.state_accuracy,
                "success_auc": score.success_auc,
                "dropout_lengths": list(score.dropouts.lengths),
                "contrast": CONTRAST[sequence.name],
            })
    finally:
        tracker.reset(); del tracker
        gc.collect(); torch.cuda.empty_cache()
    return output

def weighted(rows, key):
    frames = sum(row["frames"] for row in rows)
    return sum(row[key] * row["frames"] for row in rows) / max(frames, 1)

def summary(rows):
    return {"state_accuracy": weighted(rows, "state_accuracy"),
            "success_auc": weighted(rows, "success_auc"),
            "lost_frames": sum(sum(row["dropout_lengths"]) for row in rows),
            "longest": max((max(row["dropout_lengths"], default=0)
                            for row in rows), default=0)}

VAL_SEQUENCES = eval_subset(SPLITS["val"])
VAL_ROWS = {label: run_arm(label, VAL_SEQUENCES) for label in CONFIGS}
print(f"\n{'arm':<26}{'source':<14}{'SA':>9}{'AUC':>9}{'lost':>9}{'longest':>10}")
for label, rows in VAL_ROWS.items():
    for source in sorted({row["source"] for row in rows}):
        stats = summary([row for row in rows if row["source"] == source])
        print(f"{label:<26}{source:<14}{stats['state_accuracy']:>9.4f}"
              f"{stats['success_auc']:>9.4f}{stats['lost_frames']:>9}"
              f"{stats['longest']:>10}")
SELECTED_LABEL = max(
    ("aerial_temporal", "aerial_temporal+samurai"),
    key=lambda label: summary(VAL_ROWS[label])["state_accuracy"])
print("selected on validation:", SELECTED_LABEL)

In [ ]:
TEST_SEQUENCES = eval_subset(SPLITS["test"])
TEST_ROWS = {
    "stage_b": run_arm("stage_b", TEST_SEQUENCES),
    SELECTED_LABEL: run_arm(SELECTED_LABEL, TEST_SEQUENCES),
}
print("\nheld-out test")
for label, rows in TEST_ROWS.items():
    print(label, summary(rows))

deploy_config = WORK / "edgetam_aerial_thermal_tracking_512.yaml"
relative_checkpoint = "checkpoints/edgetam_aerial_thermal_tracking_512.pt"
deploy = tracker_config(relative_checkpoint,
                        SAMURAI if SELECTED_LABEL.endswith("+samurai") else None)
deploy_config.write_text(yaml.safe_dump(deploy, sort_keys=False))
report_file = WORK / "aerial_tracking_log.json"
report_file.write_text(json.dumps({
    "base_stage_b": str(BASE_STAGE_B), "checkpoint": str(CHECKPOINT),
    "stage_c_precheck": PRECHECK_REPORT,
    "selected_on_val": SELECTED_LABEL, "source_weights": SOURCE_WEIGHTS,
    "modality": MODALITY, "base_stage_b_tag": BASE_TAG,
    "vtuav_vis_train_parts": VTUAV_VIS_TRAIN_PARTS,
    "vtuav_vis_hold_out_parts": VTUAV_VIS_HOLD_OUT_PARTS,
    "absent_frames": ABSENT_FRAMES,
    "mask_frames": MASK_FRAMES,
    "mask_frames": MASK_FRAMES, "visible_frames": VISIBLE_FRAMES,
    "drawn_mask_frames": sum(len(store) for store in VIS_STORES.values()),
    "hold_out": VTUAV_VIS_HOLD_OUT_PARTS,
    "contrast": CONTRAST, "training": result,
    "validation": VAL_ROWS, "test": TEST_ROWS,
}, indent=2) + "\n")

shutil.copy2(CHECKPOINT, MIRROR / CHECKPOINT.name)
shutil.copy2(deploy_config, MIRROR / deploy_config.name)
shutil.copy2(report_file, MIRROR / report_file.name)
print("saved to", MIRROR)

## Before / after videoları

Validation'da ölçülmüş dizilerden iki vaka: düşük-kontrast alt grubunda en iyi
kazanç ve kaydedilmiş **en kötü gerileme**. Yeşil ground truth, kırmızı taban
(35), camgöbeği seçilmiş final koldur. İki model aynı ilk kutuyu ve aynı sabit
crop'u kullanır.

En kötü gerilemenin de gösterilmesi bilinçli: ortalama iyileşen bir koşu tek
tek dizilerde çökebilir ve karar o dizilerde verilir.

In [ ]:
from IPython.display import Video, display
from src.accuracy import box_from_mask, per_frame_iou
from src.prompts import BoxPrompt, PromptSet
from tools.eval_antiuav import _prepare_crop

def choose_video_cases():
    base = {row["name"]: row for row in VAL_ROWS["stage_b"]}
    final = {row["name"]: row for row in VAL_ROWS[SELECTED_LABEL]}
    cases = {}
    for source in sorted({row["source"] for row in base.values()}):
        names = [name for name, row in base.items()
                 if row["source"] == source and name in final]
        names.sort(key=lambda name: float(base[name]["contrast"]))
        low = names[:max(1, math.ceil(len(names) * 0.25))]
        delta = lambda name: (final[name]["state_accuracy"] -
                              base[name]["state_accuracy"])
        best, worst = max(low, key=delta), min(names, key=delta)
        cases[f"{source}_low_contrast_best"] = best
        if worst != best:
            cases[f"{source}_worst_regression"] = worst
    return cases

def hardest_segment(sequence):
    visible = sequence.labels.visible_indices()[::5]
    measured = [(int(index), frame_local_contrast(sequence, int(index)))
                for index in visible]
    measured = [(index, value) for index, value in measured
                if np.isfinite(value)]
    if measured:
        hard_index, hard_value = min(measured, key=lambda pair: pair[1])
    else:
        hard_index, hard_value = int(visible[0]), float("nan")
    begin = max(0, hard_index - DEMO_PRE_ROLL)
    end = min(len(sequence), begin + DEMO_FRAMES)
    begin = max(0, end - DEMO_FRAMES)
    return Sequence(
        name=f"{sequence.name}__demo", split=sequence.split,
        frames=sequence.frames[begin:end],
        labels=SequenceLabels(
            exist=sequence.labels.exist[begin:end].copy(),
            boxes=sequence.labels.boxes[begin:end].copy())), begin, hard_index, hard_value

def visual_track(config, segment, frames_dir, gt):
    start = int(segment.labels.visible_indices()[0])
    tracker = build_tracker("edgetam", **config)
    pred = np.full_like(gt, np.nan)
    masks = [None] * len(gt)
    try:
        tracker.prepare(frames_dir)
        tracker.set_prompts(PromptSet(boxes=[BoxPrompt(
            obj_id=1, frame_idx=start,
            xyxy=tuple(float(value) for value in gt[start]))]))
        for result_row in tracker.propagate():
            if not 0 <= result_row.frame_idx < len(gt):
                continue
            mask = result_row.masks.get(1)
            if mask is not None:
                mask = np.asarray(mask, dtype=bool)
                masks[result_row.frame_idx] = mask
                pred[result_row.frame_idx] = box_from_mask(mask)
    finally:
        tracker.reset(); del tracker
        gc.collect(); torch.cuda.empty_cache()
    return pred, masks, start

def draw_box(image, box, color):
    if np.isfinite(box).all():
        x0, y0, x1, y1 = (int(round(value)) for value in box)
        cv2.rectangle(image, (x0, y0), (x1, y1), color, 2)

def panel(gray, mask, pred, gt, title, color, iou, source_frame):
    image = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    if mask is not None and mask.shape == gray.shape:
        tint = np.zeros_like(image); tint[mask] = color
        image = cv2.addWeighted(image, 1.0, tint, 0.32, 0)
    draw_box(image, gt, (70, 220, 70)); draw_box(image, pred, color)
    cv2.rectangle(image, (0, 0), (SIZE, 52), (16, 16, 16), -1)
    cv2.putText(image, title, (12, 21), cv2.FONT_HERSHEY_SIMPLEX,
                0.58, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(image, f"IoU {iou:.3f} | frame {source_frame}", (12, 43),
                cv2.FONT_HERSHEY_SIMPLEX, 0.47, (230, 230, 230), 1,
                cv2.LINE_AA)
    return image

def render_case(case, sequence):
    segment, source_begin, hard_index, hard_value = hardest_segment(sequence)
    frames_dir = WORK / "demo_frames" / case
    if frames_dir.exists():
        shutil.rmtree(frames_dir)
    frames_dir.mkdir(parents=True)
    frames_dir, gt = _prepare_crop(segment, SIZE, frames_dir)
    before_pred, before_masks, before_start = visual_track(
        CONFIGS["stage_b"], segment, frames_dir, gt)
    after_pred, after_masks, after_start = visual_track(
        CONFIGS[SELECTED_LABEL], segment, frames_dir, gt)
    start = max(before_start, after_start)
    before_iou, after_iou = per_frame_iou(before_pred, gt), per_frame_iou(after_pred, gt)

    raw = WORK / f"{case}_raw.mp4"
    output = WORK / f"{case}_before_after.mp4"
    writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"),
                             DEMO_FPS, (2 * SIZE, SIZE))
    assert writer.isOpened()
    try:
        for index in range(start, len(gt)):
            gray = cv2.imread(str(frames_dir / f"{index}.jpg"),
                              cv2.IMREAD_GRAYSCALE)
            assert gray is not None, f"missing rendered frame {index}"
            left = panel(gray, before_masks[index], before_pred[index], gt[index],
                         "BEFORE - Stage B", (60, 70, 235),
                         before_iou[index], source_begin + index)
            right = panel(gray, after_masks[index], after_pred[index], gt[index],
                          f"AFTER - {SELECTED_LABEL}", (230, 190, 40),
                          after_iou[index], source_begin + index)
            writer.write(np.concatenate([left, right], axis=1))
    finally:
        writer.release()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw),
                    "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p",
                    str(output)], check=True)
    raw.unlink(missing_ok=True)
    target = MIRROR / "before_after_demo" / output.name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(output, target)
    return {"case": case, "sequence": sequence.name,
            "source_range": [source_begin, source_begin + len(segment)],
            "hard_frame": hard_index, "hard_contrast": hard_value,
            "before_mean_iou": float(before_iou[start:].mean()),
            "after_mean_iou": float(after_iou[start:].mean()),
            "video": str(output), "drive_video": str(target)}

VIDEO_RESULTS = []
if RENDER_BEFORE_AFTER:
    by_name = {sequence.name: sequence for sequence in sequences}
    for case, name in choose_video_cases().items():
        print("rendering", case, name)
        row = render_case(case, by_name[name])
        VIDEO_RESULTS.append(row)
        print(row)
        display(Video(row["video"], embed=True, width=1024))
    video_report = MIRROR / "before_after_demo" / "video_report.json"
    video_report.write_text(json.dumps(VIDEO_RESULTS, indent=2) + "\n")
    print("videos saved to", video_report.parent)

## Karar

**Önce stock satırına bakın.** EdgeTAM RGB videoda eğitildi, yani burada alan
içinde. Termalde stock'u yenmek alan dışı bir modeli yenmekti; RGB'de öyle
değil. Stock'a yakın durmak bile 35 + Stage C zincirinin RGB'yi bozmadığı
anlamına gelir; yenmek gerçek bir sonuçtur.

Sonra held-out tablosuna bakın — `test_00x` yazarların seçtiği dizilerdir,
hash'in değil. Test gerilerse checkpoint'i deploy etmeyin: validation'da iyi
görünen tek bir video karar değildir.

**Bu koşuda kaybolma denetimi yok.** VTUAV-VIS `exist`'i her karede True, yani
object-score kafası hiçbir şey öğrenmiyor ve kaybolup geri gelme davranışı
burada ne eğitiliyor ne ölçülüyor. O soru 31'in `vtuav` kolunda yaşar.

Bu model bir sınıf dedektörü değildir: ilk prompt kutusundan sonra herhangi bir
instance maskesini takip eden EdgeTAM'dır. Stage C kimliği sınıf adı yerine
zaman içindeki aynı nesne olarak öğrenir.